In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

dx = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_st_test_corpus_data.parquet")

dx.head()


,query,context,type,synthesized,source,metadata,url1
727493,SHEBA Ice Camp environmental monitoring,Description: NCAR portable automated mesonet (...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214601988-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
489836,bio-geochemical cycles Ross Sea,Description: The data sets include measurement...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214593766-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
869447,Pearl Harbor oceanographic data,Description: This dataset contains oceanograph...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089378855-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
668676,optical backscatter measurement techniques,Description: Two hydrographic surveys were per...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214155000-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
723711,gravity and magnetic field data integration,Description: This data set contains underway g...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214611760-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...


In [8]:
# tokenize the query and context and add number of tokens to columns
from transformers import AutoTokenizer

def tokenize_and_count_tokens(df, query_col, context_col, tokenizer):
    # Ensure all values are strings before tokenizing
    # This prevents errors from NaN, None, or other types
    df[query_col + "_tokens"] = df[query_col].apply(lambda x: tokenizer.tokenize(str(x)))
    df[context_col + "_tokens"] = df[context_col].apply(lambda x: tokenizer.tokenize(str(x)))

    # The rest of your function remains the same
    df[query_col + "_num_tokens"] = df[query_col + "_tokens"].apply(len)
    df[context_col + "_num_tokens"] = df[context_col + "_tokens"].apply(len)

    return df


tokenizer = AutoTokenizer.from_pretrained("nasa-impact/indus-sde-st-v0.2")
dx = tokenize_and_count_tokens(dx, "query", "context", tokenizer)

Token indices sequence length is longer than the specified maximum sequence length for this model (1099 > 1024). Running this sequence through the model will result in indexing errors


In [9]:
dx.head()

,query,context,type,synthesized,source,metadata,url1,query_tokens,context_tokens,query_num_tokens,context_num_tokens
727493,SHEBA Ice Camp environmental monitoring,Description: NCAR portable automated mesonet (...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214601988-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...,"[she, ba, Ġice, Ġcamp, Ġenvironmental, Ġmonito...","[description, :, Ġn, car, Ġportable, Ġautomate...",6,89
489836,bio-geochemical cycles Ross Sea,Description: The data sets include measurement...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214593766-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...,"[bio, -, ge, ochemical, Ġcycles, Ġross, Ġsea]","[description, :, Ġthe, Ġdata, Ġsets, Ġinclude,...",7,277
869447,Pearl Harbor oceanographic data,Description: This dataset contains oceanograph...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089378855-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...,"[pear, l, Ġharbor, Ġocean, ographic, Ġdata]","[description, :, Ġthis, Ġdataset, Ġcontains, Ġ...",6,357
668676,optical backscatter measurement techniques,Description: Two hydrographic surveys were per...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214155000-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...,"[optical, Ġbackscatter, Ġmeasurement, Ġtechniq...","[description, :, Ġtwo, Ġhydro, graphic, Ġsurve...",4,398
723711,gravity and magnetic field data integration,Description: This data set contains underway g...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214611760-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...,"[gravity, Ġand, Ġmagnetic, Ġfield, Ġdata, Ġint...","[description, :, Ġthis, Ġdata, Ġset, Ġcontains...",6,664


In [14]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Distribution plot for query_num_tokens
plt.figure(figsize=(10, 6))
sns.histplot(dx['query_num_tokens'], kde=True)
plt.title('Distribution of Query Number of Tokens')
plt.xlabel('Query Number of Tokens')
plt.ylabel('Frequency')
plt.grid(True)
plt.savefig('query_num_tokens_dist.png')
plt.close()

# Distribution plot for context_num_tokens
plt.figure(figsize=(10, 6))
sns.histplot(dx['context_num_tokens'], kde=True)
plt.title('Distribution of Context Number of Tokens')
plt.xlabel('Context Number of Tokens')
plt.ylabel('Frequency')
plt.grid(True)
plt.savefig('context_num_tokens_dist.png')
plt.close()



context_gt_512 = (dx['context_num_tokens'] > 512).sum()
total_data_points = len(dx)
percentage = (context_gt_512 / total_data_points) * 100
print(f"Percentage of context data points with total tokens > 512: {percentage:.2f}%")

# simil;arly for query
query_gt_512 = (dx['query_num_tokens'] > 512).sum()
total_data_points = len(dx)
percentage = (query_gt_512 / total_data_points) * 100
print(f"Percentage of query data points with total tokens > 512: {percentage:.2f}%")

Percentage of context data points with total tokens > 512: 19.65%
Percentage of query data points with total tokens > 512: 0.00%


In [16]:
dx.sort_values(by='context_num_tokens', ascending=False, inplace=True)

In [19]:
dx

,query,context,type,synthesized,source,metadata,url1,query_tokens,context_tokens,query_num_tokens,context_num_tokens
301262,normalized counts data OSD-21,"{""hits"":23,""input"":""137,87-95,13,20-50"",""page_...",search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/GENELAB_Publications_Website/|htt...","https://osdr.nasa.gov/osdr/data/osd/files/137,...","[normalized, Ġcounts, Ġdata, Ġos, d, -, 21]","[{, "", h, its, "":, 23, ,"", input, "":, "", 137, ...",7,158035
301263,largest file in OSD-30,"{""hits"":23,""input"":""137,87-95,13,20-50"",""page_...",search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/GENELAB_Publications_Website/|htt...","https://osdr.nasa.gov/osdr/data/osd/files/137,...","[largest, Ġfile, Ġin, Ġos, d, -, 30]","[{, "", h, its, "":, 23, ,"", input, "":, "", 137, ...",7,158035
301260,GLDS-13 array processing,"{""hits"":23,""input"":""137,87-95,13,20-50"",""page_...",search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/GENELAB_Publications_Website/|htt...","https://osdr.nasa.gov/osdr/data/osd/files/137,...","[gl, ds, -, 13, Ġarray, Ġprocessing]","[{, "", h, its, "":, 23, ,"", input, "":, "", 137, ...",6,158035
300933,user consent handling GTM,// Copyright 2012 Google Inc. All rights reser...,search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/GENELAB_Publications_Website/|htt...",https://www.googletagmanager.com/gtag/js?id=G-...,"[user, Ġconsent, Ġhandling, Ġg, tm]","[//, Ġcopyright, Ġ2012, Ġgoogle, Ġinc, ., Ġall...",5,142354
300937,GTM enhanced conversions,// Copyright 2012 Google Inc. All rights reser...,search_term-document,True,SDE_general_v3,"{""id"": ""/SDE/GENELAB_Publications_Website/|htt...",https://www.googletagmanager.com/gtag/js?id=G-...,"[g, tm, Ġenhanced, Ġconversions]","[//, Ġcopyright, Ġ2012, Ġgoogle, Ġinc, ., Ġall...",4,142354
...,...,...,...,...,...,...,...,...,...,...,...
13854,What is the date of the Astronomy Picture of t...,2018 June 6,question-answer,True,SDE_general_v2,"{""id"": ""/SDE/astronomy_picture_of_the_day/|htt...",https://apod.nasa.gov/apod/ap180606.html,"[what, Ġis, Ġthe, Ġdate, Ġof, Ġthe, Ġastronomy...","[2018, Ġjune, Ġ6]",16,3
49833,What is the main focus of the 'Astrobiology St...,Astrobiology Strategy,question-answer,True,SDE_general_v2,"{""id"": ""/SDE/astrobiology_at_nasa/|https://ast...",https://astrobiology.nasa.gov/nai/directory/ro...,"[what, Ġis, Ġthe, Ġmain, Ġfocus, Ġof, Ġthe, Ġ'...","[astro, biology, Ġstrategy]",17,3
76708,How can one contact NASA for inquiries regardi...,Contact NASA,question-answer,True,SDE_general_v2,"{""id"": ""/SDE/neil_gehrel_s_swift_observatory/|...",https://swift.gsfc.nasa.gov/results/bs70mon/SW...,"[how, Ġcan, Ġone, Ġcontact, Ġnasa, Ġfor, Ġinqu...","[contact, Ġnasa]",11,2
72255,What type of data can be requested from the LROC?,Target Request,question-answer,True,SDE_general_v2,"{""id"": ""/SDE/lunar_reconnaissance_orbiter_came...",https://www.lroc.asu.edu/atlases/psr/NP_858100...,"[what, Ġtype, Ġof, Ġdata, Ġcan, Ġbe, Ġrequeste...","[target, Ġrequest]",12,2


In [22]:
dx[(dx["context_num_tokens"] > 512)].shape, dx[(dx["context_num_tokens"] > 512) & (dx["context_num_tokens"] < 1024)].shape

((25246, 11), (19064, 11))

In [23]:
dx_filt = dx[(dx["context_num_tokens"] > 512) & (dx["context_num_tokens"] < 1024)]


In [24]:
dx_filt.to_parquet("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_st_test_long_corpus_data.parquet")